# Blackjack Simulator — Variance Analysis

## Objective

This notebook measures the **variance of simulation outcomes across different random seeds**.

Each seed produces a fully independent run of the same configuration. By comparing results across seeds, we can:
- Quantify how much EV% fluctuates run-to-run for a given number of hands
- Visualise the convergence of the mean EV as sample size grows
- Evaluate the impact of different bet spreads on expected value and variance

The simulation uses **basic strategy only** (6 decks, configurable rules).

---

## 1. Imports

In [ ]:
import subprocess, sys

# Install required packages if missing
_packages = ["pandas", "matplotlib"]
for _pkg in _packages:
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", _pkg, "--quiet"])

import os

# Make sure the project root is on the path when running from notebooks/
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from simulation.simulator import simulate
from simulation.config import BettingConfig, SimConfig, TableRules

## 2. Parameters

Edit the values in this cell to configure the analysis.

In [ ]:
# ── Simulation parameters ────────────────────────────────────────────────────
HANDS        = 1_000_000   # number of hands per seed
SEEDS        = range(1, 11) # seeds to run (10 independent runs)
SPREAD       = "1-1"        # bet spread: "MIN-MAX" (e.g. "1-1", "1-12")
UNIT_SIZE    = 5.0          # value of one betting unit (EUR)
BANKROLL     = 10_000_000.0 # initial bankroll (large enough to never bust)

# ── Shoe parameters ──────────────────────────────────────────────────────────
DECKS        = 6
PENETRATION  = 0.75         # fraction of shoe dealt before reshuffle

# ── Table rules ──────────────────────────────────────────────────────────────
DEALER_H17   = False        # False = S17 (stands on soft 17), True = H17
DAS          = True         # double after split
SURRENDER    = True         # late surrender allowed
BJ_PAYOUT    = 1.5          # blackjack payout (1.5 = 3:2)

## 3. Simulation

Run one simulation per seed and collect results into a DataFrame.

In [ ]:
def run_seed(seed: int) -> dict:
    """Run a single simulation and return a flat dict of key metrics."""
    cfg = SimConfig(
        hands            = HANDS,
        decks            = DECKS,
        penetration      = PENETRATION,
        betting          = BettingConfig.from_string(SPREAD, unit_size=UNIT_SIZE),
        rules            = TableRules(
            dealer_hits_soft_17 = DEALER_H17,
            double_after_split  = DAS,
            surrender_allowed   = SURRENDER,
            blackjack_payout    = BJ_PAYOUT,
        ),
        initial_bankroll = BANKROLL,
        seed             = seed,
    )
    r = simulate(cfg)
    return {
        "seed"             : seed,
        "ev_percent"       : r.ev_percent,
        "ev_per_100_hands" : r.ev_per_100_hands,
        "total_profit"     : r.total_profit,
        "final_bankroll"   : r.final_bankroll,
        "average_bet"      : r.average_bet,
        "wins"             : r.wins,
        "losses"           : r.losses,
        "pushes"           : r.pushes,
        "positive_tc_ratio": r.positive_tc_ratio,
        "profit_std"       : r.profit_std,
    }


print(f"Running {len(list(SEEDS))} simulations × {HANDS:,} hands each …")

records = []
for seed in SEEDS:
    rec = run_seed(seed)
    records.append(rec)
    print(f"  seed={seed:>3d}  EV={rec['ev_percent']:+.4f}%  profit={rec['total_profit']:+,.0f} EUR")

df = pd.DataFrame(records).set_index("seed")
print(f"\nDone. DataFrame shape: {df.shape}")
df

## 4. Statistical Analysis

In [ ]:
# Summary statistics for the most relevant financial metrics
metrics_of_interest = ["ev_percent", "ev_per_100_hands", "total_profit", "final_bankroll"]

summary = (
    df[metrics_of_interest]
    .agg(["mean", "std", "min", "max"])
    .T
    .rename(columns={"mean": "Mean", "std": "Std Dev", "min": "Min", "max": "Max"})
)

# Format nicely
pd.set_option("display.float_format", "{:+.4f}".format)
print("=" * 60)
print(f"  Variance summary — {len(list(SEEDS))} seeds × {HANDS:,} hands")
print(f"  Spread: {SPREAD}  |  Unit: {UNIT_SIZE} EUR  |  Rules: {'H17' if DEALER_H17 else 'S17'}, "
      f"{'DAS' if DAS else 'noDAS'}, {'SUR' if SURRENDER else 'noSUR'}")
print("=" * 60)
print(summary.to_string())
print("=" * 60)

# Reset display format
pd.reset_option("display.float_format")

## 5. Charts

### 5.1 EV% per seed (line plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df.index, df["ev_percent"], marker="o", linewidth=1.5)
ax.axhline(df["ev_percent"].mean(), linestyle="--", linewidth=1, label=f"Mean = {df['ev_percent'].mean():+.4f}%")
ax.axhline(0, linestyle=":", linewidth=0.8)

ax.set_title(f"EV% per seed — {HANDS:,} hands, spread {SPREAD}")
ax.set_xlabel("Seed")
ax.set_ylabel("EV (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%+.3f%%"))
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

### 5.2 Total profit per seed (bar plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(df.index, df["total_profit"])
ax.axhline(0, linewidth=0.8)
ax.axhline(df["total_profit"].mean(), linestyle="--", linewidth=1,
           label=f"Mean = {df['total_profit'].mean():+,.0f} EUR")

ax.set_title(f"Total profit per seed — {HANDS:,} hands, spread {SPREAD}")
ax.set_xlabel("Seed")
ax.set_ylabel("Profit (EUR)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:+,.0f}"))
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

### 5.3 Final bankroll per seed (bar plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(df.index, df["final_bankroll"])
ax.axhline(BANKROLL, linestyle="--", linewidth=1, label=f"Initial bankroll = {BANKROLL:,.0f} EUR")
ax.axhline(df["final_bankroll"].mean(), linestyle=":", linewidth=1,
           label=f"Mean = {df['final_bankroll'].mean():,.0f} EUR")

ax.set_title(f"Final bankroll per seed — {HANDS:,} hands, spread {SPREAD}")
ax.set_xlabel("Seed")
ax.set_ylabel("Bankroll (EUR)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

### 5.4 Distribution of EV% (histogram)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(df["ev_percent"], bins="auto", edgecolor="white")
ax.axvline(df["ev_percent"].mean(), linestyle="--", linewidth=1.5,
           label=f"Mean = {df['ev_percent'].mean():+.4f}%")
ax.axvline(0, linestyle=":", linewidth=1)

ax.set_title(f"Distribution of EV% — {len(list(SEEDS))} seeds")
ax.set_xlabel("EV (%)")
ax.set_ylabel("Count")
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%+.3f%%"))
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

### 5.5 EV% convergence — expanding mean across seeds

In [ ]:
expanding_mean = df["ev_percent"].expanding().mean()

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df.index, df["ev_percent"], marker="o", linewidth=0.8,
        linestyle="--", label="EV% per seed")
ax.plot(expanding_mean.index, expanding_mean, marker="s", linewidth=2,
        label="Expanding mean")
ax.axhline(0, linestyle=":", linewidth=0.8)

ax.set_title(f"Convergence of mean EV% as seeds accumulate — spread {SPREAD}")
ax.set_xlabel("Seeds included")
ax.set_ylabel("EV (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%+.4f%%"))
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()

---

## 6. Bonus — Spread Comparison

Run the same seeds across multiple bet spreads and compare EV% and profit.

> **Note:** This section runs additional simulations and may take several minutes.

In [ ]:
SPREADS_TO_COMPARE = ["1-1", "1-4", "1-8", "1-12"]
SEEDS_BONUS        = range(1, 6)   # fewer seeds to keep runtime manageable
HANDS_BONUS        = 500_000

spread_records = []

for spread in SPREADS_TO_COMPARE:
    for seed in SEEDS_BONUS:
        cfg = SimConfig(
            hands            = HANDS_BONUS,
            decks            = DECKS,
            penetration      = PENETRATION,
            betting          = BettingConfig.from_string(spread, unit_size=UNIT_SIZE),
            rules            = TableRules(
                dealer_hits_soft_17 = DEALER_H17,
                double_after_split  = DAS,
                surrender_allowed   = SURRENDER,
                blackjack_payout    = BJ_PAYOUT,
            ),
            initial_bankroll = BANKROLL,
            seed             = seed,
        )
        r = simulate(cfg)
        spread_records.append({
            "spread"           : spread,
            "seed"             : seed,
            "ev_percent"       : r.ev_percent,
            "ev_per_100_hands" : r.ev_per_100_hands,
            "total_profit"     : r.total_profit,
            "average_bet"      : r.average_bet,
            "profit_std"       : r.profit_std,
        })
    print(f"  spread {spread} done.")

df_spread = pd.DataFrame(spread_records)
print(f"\nTotal runs: {len(df_spread)}")

In [ ]:
# Grouped summary by spread
spread_summary = (
    df_spread
    .groupby("spread")[["ev_percent", "ev_per_100_hands", "total_profit", "average_bet", "profit_std"]]
    .agg(["mean", "std"])
)

pd.set_option("display.float_format", "{:.4f}".format)
print("Spread comparison — mean and std across seeds")
print(spread_summary.to_string())
pd.reset_option("display.float_format")

In [ ]:
# ── EV% mean ± 1σ per spread ─────────────────────────────────────────────────
ev_mean = df_spread.groupby("spread")["ev_percent"].mean()
ev_std  = df_spread.groupby("spread")["ev_percent"].std()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: EV% with error bars
ax = axes[0]
ax.bar(ev_mean.index, ev_mean, yerr=ev_std, capsize=4)
ax.axhline(0, linewidth=0.8)
ax.set_title("EV% by spread (mean ± 1σ)")
ax.set_xlabel("Spread")
ax.set_ylabel("EV (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%+.3f%%"))
ax.grid(axis="y", linestyle=":", alpha=0.6)

# Right: average bet per spread
ax = axes[1]
avg_bet_mean = df_spread.groupby("spread")["average_bet"].mean()
ax.bar(avg_bet_mean.index, avg_bet_mean)
ax.set_title("Average bet by spread")
ax.set_xlabel("Spread")
ax.set_ylabel("Average bet (EUR)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.2f}"))
ax.grid(axis="y", linestyle=":", alpha=0.6)

plt.suptitle(f"Spread comparison — {HANDS_BONUS:,} hands × {len(list(SEEDS_BONUS))} seeds",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# EV% box plot per spread
spreads_order = SPREADS_TO_COMPARE
data_per_spread = [df_spread[df_spread["spread"] == s]["ev_percent"].values for s in spreads_order]

fig, ax = plt.subplots(figsize=(9, 5))

ax.boxplot(data_per_spread, labels=spreads_order, patch_artist=False)
ax.axhline(0, linestyle=":", linewidth=0.8)

ax.set_title(f"EV% distribution by spread — {HANDS_BONUS:,} hands")
ax.set_xlabel("Spread")
ax.set_ylabel("EV (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%+.3f%%"))
ax.grid(axis="y", linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()